# <center>  Описание проекта:</center>

Наша компания купила крупный сервис для чтения книг по подписке.
Наша задача как аналитика — проанализировать базу данных.
В ней — информация о книгах, издательствах, авторах, а также пользовательские обзоры книг. 
Эти данные помогут сформулировать ценностное предложение для нового продукта.

# <center>  Описание данных:</center>



**Таблица `books`**
Содержит данные о книгах:
- `book_id` — идентификатор книги;
- `author_id` — идентификатор автора;
- `title` — название книги;
- `num_pages` — количество страниц;
- `publication_date` — дата публикации книги;
- `publisher_id` — идентификатор издателя.

**Таблица `authors`**
Содержит данные об авторах:
- `author_id` — идентификатор автора;
- `author` — имя автора.

**Таблица `publishers`**
Содержит данные об издательствах:
- `publisher_id` — идентификатор издательства;
- `publisher` — название издательства.

**Таблица `ratings`**
Содержит данные о пользовательских оценках книг:
- `rating_id` — идентификатор оценки;
- `book_id` — идентификатор книги;
- `username` — имя пользователя, оставившего оценку;
- `rating` — оценка книги.

**Таблица `reviews`**
Содержит данные о пользовательских обзорах:
- `review_id` — идентификатор обзора;
- `book_id` — идентификатор книги;
- `username` — имя автора обзора;
- `text` — текст обзора.


In [1]:
# импортируем библиотеки
import pandas as pd
import sqlalchemy as sa

In [2]:
# устанавливаем параметры
db_config = {
'user': 'praktikum_student', # имя пользователя
'pwd': 'Sdf4$2;d-d30pp', # пароль
'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
'port': 6432, # порт подключения
'db': 'data-analyst-final-project-db' # название базы данных
}
connection_string = 'postgresql://{user}:{pwd}@{host}:{port}/{db}'.format(**db_config)

In [3]:
# сохраняем коннектор
engine = sa.create_engine(connection_string, connect_args={'sslmode':'require'})

In [4]:
# чтобы выполнить SQL-запрос, пишем функцию с использованием Pandas
def get_sql_data(query:str, engine:sa.engine.base.Engine=engine) -> pd.DataFrame:
    '''Открываем соединение, получаем данные из sql, закрываем соединение'''
    with engine.connect() as con:
        return pd.read_sql(sql=sa.text(query), con = con)

## Выводим первые 5 строк и считаем количество строк в каждой таблице

In [5]:
query = '''SELECT * FROM books LIMIT 5'''
get_sql_data(query)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


In [6]:
query = '''SELECT COUNT(*) FROM books LIMIT 5'''
get_sql_data(query)

,count
0,1000


In [7]:
query = '''SELECT * FROM authors LIMIT 5'''
get_sql_data(query)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


In [8]:
query = '''SELECT COUNT(*) FROM authors LIMIT 5'''
get_sql_data(query)

,count
0,636


In [9]:
query = '''SELECT * FROM publishers LIMIT 5'''
get_sql_data(query)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


In [10]:
query = '''SELECT COUNT(*) FROM publishers LIMIT 5'''
get_sql_data(query)

,count
0,340


In [11]:
query = '''SELECT * FROM ratings LIMIT 5'''
get_sql_data(query)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


In [12]:
query = '''SELECT COUNT(*) FROM ratings LIMIT 5'''
get_sql_data(query)

,count
0,6456


In [13]:
query = '''SELECT * FROM reviews LIMIT 5'''
get_sql_data(query)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


In [14]:
query = '''SELECT COUNT(*) FROM reviews LIMIT 5'''
get_sql_data(query)

,count
0,2793


## SQL-запросы для решения заданий.

###  Посчитайте, сколько книг вышло после 1 января 2000 года;

In [15]:
# Посчитайте, сколько книг вышло после 1 января 2000 года

query = '''
SELECT COUNT(*) AS count_books
FROM books
WHERE publication_date > '2000-01-01'
'''
get_sql_data(query)

,count_books
0,819


После 1 января 2000 года вышло 819 книг.

### Для каждой книги посчитайте количество обзоров и среднюю оценку;

In [20]:
# Для каждой книги посчитайте количество обзоров и среднюю оценку

query = '''

SELECT books.book_id,books.title, COUNT(DISTINCT reviews.review_id) AS count_review, AVG(ratings.rating) AS avg_rating
FROM books
LEFT JOIN reviews ON books.book_id=reviews.book_id
LEFT JOIN ratings ON books.book_id=ratings.book_id
GROUP BY books.book_id
ORDER BY books.book_id, books.title 
'''
get_sql_data(query)

,book_id,title,count_review,avg_rating
0,1,'Salem's Lot,2,3.666667
1,2,1 000 Places to See Before You Die,1,2.500000
2,3,13 Little Blue Envelopes (Little Blue Envelope...,3,4.666667
3,4,1491: New Revelations of the Americas Before C...,2,4.500000
4,5,1776,4,4.000000
...,...,...,...,...
995,996,Wyrd Sisters (Discworld #6; Witches #2),3,3.666667
996,997,Xenocide (Ender's Saga #3),3,3.400000
997,998,Year of Wonders,4,3.200000
998,999,You Suck (A Love Story #2),2,4.500000


In [22]:
# посчитал минимальные и максимальные значения для выводовю
query = '''
SELECT MAX(count_review), MIN(count_review), MAX(avg_rating), MIN(avg_rating)
FROM(
SELECT books.book_id,books.title, COUNT(DISTINCT reviews.review_id) AS count_review, AVG(ratings.rating) AS avg_rating
FROM books
LEFT JOIN reviews ON books.book_id=reviews.book_id
LEFT JOIN ratings ON books.book_id=ratings.book_id
GROUP BY books.book_id
ORDER BY books.book_id, books.title) AS t1
'''
get_sql_data(query)

,max,min,max,min
0,7,0,5.0,1.5


###  Определите издательство, которое выпустило наибольшее число книг толще 50 страниц — так вы исключите из анализа брошюры;

In [17]:
# Определите издательство, которое выпустило наибольшее число книг толще 50 страниц — так вы исключите из анализа брошюры

query = '''
SELECT publishers.publisher,COUNT(t1.book_id)
FROM publishers
JOIN 
(SELECT *
FROM books
WHERE num_pages > 50) AS t1 ON publishers.publisher_id=t1.publisher_id
GROUP BY publishers.publisher
ORDER BY COUNT(t1.book_id) DESC
LIMIT 1

'''
get_sql_data(query)

,publisher,count
0,Penguin Books,42


Издательство Penguin Books выпустило наибольшее число книг.

### Определите автора с самой высокой средней оценкой книг — учитывайте только книги с 50 и более оценками;

In [18]:
# Определите автора с самой высокой средней оценкой книг — учитывайте только книги с 50 и более оценками

query = '''

SELECT authors.author, AVG(ratings.rating)
FROM books
JOIN authors ON books.author_id=authors.author_id
JOIN ratings ON books.book_id=ratings.book_id
WHERE books.book_id IN (SELECT book_id
                  FROM ratings
                  GROUP BY book_id
                  HAVING COUNT(rating_id) >= 50)
GROUP BY authors.author
ORDER BY AVG(ratings.rating) DESC
LIMIT 1
                  
'''
get_sql_data(query)

,author,avg
0,J.K. Rowling/Mary GrandPré,4.287097


Авторы J.K. Rowling/Mary GrandPré имеют самую высокую среднюю оценку книг.

### Посчитайте среднее количество обзоров от пользователей, которые поставили больше 48 оценок.

In [19]:
# Посчитайте среднее количество обзоров от пользователей, которые поставили больше 48 оценок

query = '''

SELECT AVG(count)
FROM (SELECT reviews.username, COUNT(reviews.review_id)
FROM reviews
WHERE reviews.username IN (SELECT username
FROM ratings
GROUP BY username
HAVING COUNT(rating_id) > 48)
GROUP BY reviews.username) AS t1



'''
get_sql_data(query)

,avg
0,24.0


Среднее количетсво обзоров от пользователей, который поставили больше 48 оценок составляет 24.

## Вывод:

Результаты изучения базы данных платформы:

- **Узнать сколько книг вышло после 1 января 2000 года:**
        - После 1 января 2000 года вышло 819 книг.
- **Для каждой книги посчитать количество обзоров и среднюю оценку:**
        - Максимальное кол-во обзоров у книги - 7, минимальное - 0. Максимальный рейтинг - 5, минимальный средний рейтинг - 1.5.
- **Определите издательство, которое выпустило наибольшее число книг толще 50 страниц:**
        - Издательство Penguin Books выпустило наибольшее число книг. - 42 книги.
- **Определите автора с самой высокой средней оценкой книг в которых от 50 оценок:**
        - Авторы J.K. Rowling/Mary GrandPré имеют самую высокую среднюю оценку книг.
- **Определить среднее количество обзоров от пользователей, которые поставили больше 48 оценок:**
        - Среднее количетсво обзоров от пользователей, который поставили больше 48 оценок составляет 24.